# 03. ML Agent — 머신러닝 Tool 통합
**Day 14**

> Day 8~11에서 배운 머신러닝 모델을 **LangChain `@tool`** 로 감싸 Agent가 호출합니다.  
> 구현 코드: `ml_tools/` 폴더 (로지스틱 회귀 · XGBoost · One-Class SVM)

---
## 0. 설치 & 경로

In [ ]:
#!uv add openai python-dotenv langchain langchain-openai langchain-core langchain-classic scikit-learn xgboost joblib

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
ML_TOOLS_DIR = WORKDIR / 'ml_tools'
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

print('WORKDIR:', WORKDIR)
print('ML_TOOLS:', ML_TOOLS_DIR)
for p in sorted((ML_TOOLS_DIR / 'data').glob('*.csv')):
    print(' - data/', p.name)

WORKDIR: d:\autornd\SK Autonomous R&D\실습\14일차_실습
ML_TOOLS: d:\autornd\SK Autonomous R&D\실습\14일차_실습\ml_tools
 - data/ anomaly_sensor.csv
 - data/ failure_equipment.csv


---
## 1. Day 8~11 ML 방법론 요약

| Tool | 강의 | 알고리즘 | Agent가 쓰는 상황 |
|------|------|----------|-------------------|
| `predict_failure_logistic` | 로지스틱 회귀 | 설비 고장 Yes/No + **확률** (해석·설명) |
| `predict_failure_xgboost` | XGBoost | 불균형 고장 데이터 **예측 성능** 우선 |
| `detect_anomaly_oneclass` | One-Class SVM | 정상 기준 **이상 탐지** (2D 센서) |

공통 패턴:
1. `ml_tools/`에서 **사전 학습 모델** 로드 (`models/*.joblib`)
2. `@tool` 함수가 **문자열**로 결과 반환 → Agent가 읽고 답변

---
## Step 1. ML Tool 불러오기

In [2]:
from ml_tools import ML_TOOLS, predict_failure_logistic, predict_failure_xgboost, detect_anomaly_oneclass

print('등록된 ML Tool:')
for t in ML_TOOLS:
    print(f' - {t.name}: {t.description[:60]}...')

등록된 ML Tool:
 - predict_failure_logistic: 설비 고장(Failure) 여부를 로지스틱 회귀(Day 2-2)로 예측합니다.

Args:
    tempe...
 - predict_failure_xgboost: 설비 고장(Failure) 여부를 XGBoost(Day 3-2)로 예측합니다.

Args:
    tempe...
 - detect_anomaly_oneclass: 2차원 센서값이 정상 범위인지 One-Class SVM(Day 4-3 / SVDD)으로 탐지합니다.

Arg...


---
## ✏️ 실습 1. `@tool` 직접 만들기

**목표:** 현재 시각을 `HH:MM` 형식으로 반환하는 `get_time_hhmm` Tool을 만듭니다.

**힌트:** `from langchain_core.tools import tool` · `datetime.now().strftime('%H:%M')`

In [3]:
# ── 실습 1. get_time_hhmm Tool ──
from datetime import datetime
from langchain_core.tools import tool


@tool
def get_time_hhmm() -> str:
    """현재 시각을 HH:MM 형식으로 반환합니다."""
    return datetime.now().strftime("%H:%M")


# 예제: Tool 단독 호출
print("name       :", get_time_hhmm.name)
print("description:", get_time_hhmm.description)
print("result     :", get_time_hhmm.invoke({}))

name       : get_time_hhmm
description: 현재 시각을 HH:MM 형식으로 반환합니다.
result     : 16:17


---
## Step 2. Tool 단독 테스트

Agent 없이 `.invoke()`로 먼저 확인합니다. (14.1의 `get_current_time.invoke`와 동일)

In [4]:
# 설비 고장 예측 — 로지스틱 회귀 (Day 2-2)
print(predict_failure_logistic.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


AttributeError: 'LogisticRegression' object has no attribute 'multi_class'

In [ ]:
# 설비 고장 예측 — XGBoost (Day 3-2)
print(predict_failure_xgboost.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

In [5]:
# 이상 탐지 — One-Class SVM (Day 4-3 / SVDD)
samples = [(1.0, 1.5, '정상에 가까운 값'), (5.0, -4.0, '이상에 가까운 값')]
for f0, f1, label in samples:
    print(f'[{label}]')
    print(detect_anomaly_oneclass.invoke({'feature_0': f0, 'feature_1': f1}))
    print('-' * 40)

[정상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------
[이상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.9268
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------


d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


---
## 센서값 직접 바꿔 `.invoke()`

`detect_anomaly_oneclass`에 임의의 `feature_0`, `feature_1` 값을 넣어 결과를 확인하세요.

**관찰:** decision_score가 0에 가까울수록 경계선 샘플입니다.

In [6]:
# ── 센서값 직접 바꿔 invoke ──
# decision_score > 0 → 정상 쪽, < 0 → 이상 쪽, 0에 가까울수록 경계선

my_samples = [
    (0.5, 0.5, '중심 근처 (정상 후보)'),
    (2.0, 2.0, '경계 근처 후보'),
    (8.0, -6.0, '멀리 떨어진 이상 후보'),
]

for my_f0, my_f1, label in my_samples:
    print(f'[{label}] feature_0={my_f0}, feature_1={my_f1}')
    print(detect_anomaly_oneclass.invoke({'feature_0': my_f0, 'feature_1': my_f1}))
    print('-' * 40)

[중심 근처 (정상 후보)] feature_0=0.5, feature_1=0.5
[One-Class SVM] 판정: 이상
- decision_score: -0.4598
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------
[경계 근처 후보] feature_0=2.0, feature_1=2.0
[One-Class SVM] 판정: 정상
- decision_score: 0.0533
- raw_prediction: 1 (+1 정상 / -1 이상)
----------------------------------------
[멀리 떨어진 이상 후보] feature_0=8.0, feature_1=-6.0
[One-Class SVM] 판정: 이상
- decision_score: -0.9270
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------


d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.9.0

---
## Step 3. AgentExecutor — ML Tool 통합

14.1과 같이 `create_tool_calling_agent` + `AgentExecutor`를 씁니다.

In [8]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.1)

agent_tools = ML_TOOLS
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 제조 설비 분석을 돕는 AI입니다. 질문에 맞는 ML 도구를 골라 사용하세요.

- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost, 불균형 데이터) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass

온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## ✏️ 실습 3. 시스템 프롬프트 수정

Step 3의 `prompt` **system** 문자열에 아래 규칙을 **한 줄** 추가한 뒤, Agent를 다시 만들어 보세요.

> *"로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요."*

**힌트:** `agent` · `executor` 변수를 다시 할당하면 됩니다.

In [9]:
# ── 여기에 작성 ──
# (Step 3 prompt 복사 후 system 문자열에 규칙 1줄 추가)
# agent = create_tool_calling_agent(llm, agent_tools, prompt)
# executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

prompt = ChatPromptTemplate.from_messages([
    ('system', 
    
'''당신은 제조 설비 분석을 돕는 AI입니다. 질문에 맞는 ML 도구를 골라 사용하세요.

- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost, 불균형 데이터) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass

로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요.
온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

---
## Step 4. Agent 데모 질문

In [14]:
# 로지스틱 회귀 Tool 선택 유도
r = executor.invoke({
    'input': '온도 67, 습도 82, Operator1, 이전 고장 후 90시간 — 로지스틱으로 고장 확률 알려줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...



Invoking: `predict_failure_logistic` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%로지스틱 회귀로 예측한 결과입니다.

- **고장(Failure) 확률:** **1.98%**
- **정상(No) 확률:** **98.02%**
- **예측 클래스:** **No (정상)**

> Finished chain.
로지스틱 회귀로 예측한 결과입니다.

- **고장(Failure) 확률:** **1.98%**
- **정상(No) 확률:** **98.02%**
- **예측 클래스:** **No (정상)**


In [12]:
# XGBoost Tool 선택 유도
r = executor.invoke({
    'input': '온도 67, 습도 82, Operator1, 이전 고장 후 90시간 - XGBoost 모델 예측도 해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`




C:\Users\Admin\miniconda3\Lib\pickle.py:1760: UserWarning: [16:44:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\data\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


[XGBoost] 예측: No
- 고장(Failure) 확률: 0.74%
- 정상(No) 확률: 99.26%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)주어진 조건(온도 67, 습도 82, Operator1, 이전 고장 후 90시간)에서 **XGBoost 예측 결과**는 다음과 같습니다.

- **예측 클래스:** **No (정상)**
- **Failure 확률:** **0.74%**
- **No(정상) 확률:** **99.26%**
- 참고(주요 변수, 학습 기준): Temperature, Hours Since Previous Failure, Humidity

> Finished chain.
주어진 조건(온도 67, 습도 82, Operator1, 이전 고장 후 90시간)에서 **XGBoost 예측 결과**는 다음과 같습니다.

- **예측 클래스:** **No (정상)**
- **Failure 확률:** **0.74%**
- **No(정상) 확률:** **99.26%**
- 참고(주요 변수, 학습 기준): Temperature, Hours Since Previous Failure, Humidity


In [15]:
# One-Class SVM 이상 탐지
r = executor.invoke({
    'input': '센서 feature_0=1.0, feature_1=1.5 일 때 정상인지 이상인지 판정해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `detect_anomaly_oneclass` with `{'feature_0': 1, 'feature_1': 1.5}`


[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)주어진 센서값 **feature_0=1.0, feature_1=1.5**에 대해 One-Class SVM 판정 결과는 **이상(-1)** 입니다.  
- decision_score: **-0.0711** (값이 낮을수록 이상 쪽)

> Finished chain.
주어진 센서값 **feature_0=1.0, feature_1=1.5**에 대해 One-Class SVM 판정 결과는 **이상(-1)** 입니다.  
- decision_score: **-0.0711** (값이 낮을수록 이상 쪽)


---
## ✏️ 실습 4. Agent에 본인 질문 작성

**고장 예측** 또는 **이상 탐지** 중 하나를 골라, Step 4 데모와 다른 조건으로 `executor.invoke`를 직접 호출하세요.

**예시:** *"온도 80, 습도 90, Operator3, 이전 고장 10시간 — XGBoost로 예측해줘"*

In [17]:
# ── 여기에 작성 ──
my_question = '온도 80, 습도 90, Operator3, 이전 고장 10시간 — 로지스틱과 XGBoost로 예측하고 더 나은 모델을 알려줘, 모델에 적절한 성능지표를 선정하고 그것을 바탕으로 비교해줘'
r = executor.invoke({'input': my_question, 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 80, 'humidity': 90, 'operator': 'Operator3', 'hours_since_previous_failure': 10}`


[로지스틱 회귀] 예측: Yes
- 고장(Failure) 확률: 56.81%
- 정상(No) 확률: 43.19%
Invoking: `predict_failure_xgboost` with `{'temperature': 80, 'humidity': 90, 'operator': 'Operator3', 'hours_since_previous_failure': 10}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 6.31%
- 정상(No) 확률: 93.69%
- 주요 변수(학습 기준): Temperature(0.212), Humidity(0.140), Hours Since Previous Failure(0.110)입력값(온도 80, 습도 90, Operator3, 이전 고장 10시간)에 대해 **로지스틱 회귀**와 **XGBoost**를 각각 예측했고, 두 모델 중 “더 나은 모델”을 판단하기 위해 **적절한 성능지표**를 어떤 걸 써야 하는지도 함께 비교 기준으로 삼겠습니다.

## 1) 예측 결과 비교
### (1) 로지스틱 회귀
- **예측: Yes (고장)**
- **고장 확률: 56.81%**
- 정상 확률: 43.19%

### (2) XGBoost
- **예측: No (정상)**
- **고장 확률: 6.31%**
- 정상 확률: 93.69%

➡️ 두 모델이 **같은 입력에서 완전히 상반된 결론**을 내렸습니다. (로지스틱은 고장 쪽, XGBoost는 정상 쪽)

---

## 2) “더 나은 모델”을 고르기 위한 성능지표(권장)
고장(Failure)은 보통 **불균형(희귀 이벤트)** 인 경우가 많고, 실제 운

---
## Step 5. 멀티턴 — `chat_history`

In [18]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input'
        : question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('온도 70, 습도 75, Operator2, 이전 고장 120시간 — 고장 날까?'))
print('2:', session.ask('방금 답에서 고장 확률만 숫자로 다시 말해줘'))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%
Invoking: `predict_failure_xgboost` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[XGBoost] 예측: Yes
- 고장(Failure) 확률: 96.73%
- 정상(No) 확률: 3.27%
- 주요 변수(학습 기준): Temperature(0.212), Humidity(0.140), Hours Since Previous Failure(0.110)주어진 조건(온도 70, 습도 75, Operator2, 이전 고장 120시간)에 대해 예측 모델이 서로 다르게 나왔습니다.

- **로지스틱 회귀 결과**: **고장 아님(No)**  
  - Failure 확률 **11.09%**, Normal 확률 **88.91%**

- **XGBoost 결과**: **고장(Yes)**  
  - Failure 확률 **96.73%**, Normal 확률 **3.27%**

따라서, **어느 쪽을 더 신뢰할지**(데이터 불균형/모델 성능 기준)에 따라 결론이 달라질 수 있습니다.  
원하시면 두 모델 중 어떤 상황에서 더 우선하는지(예: 오탐/미탐 비용) 기준을 알려주세요.

> Finished chain.
1: 주어진 조건(온도 70, 습도 75, Operator2, 이전 고장 120시간)에 대해 예측 모델이 서로 다르게 나왔습니다.

- **로지스틱 회귀 결과**: *

---
## ✏️ 실습 5. 멀티턴 후속 질문

`ChatSession`으로 1턴 예측 후, **2턴에서 맥락을 이어가는** 후속 질문을 직접 작성하세요.

**예시:** 1턴 고장 예측 → 2턴 *"그럼 같은 조건으로 이상 탐지도 해줘 feature_0=2, feature_1=2"*

In [19]:
# ── 여기에 작성 ──
session2 = ChatSession(executor)
q1 = '온도 70, 습도 75, Operator2, 이전 고장 120시간 — 고장 날까?, 로지스틱, XGBoost 결과를 알려줘'
q2 = '두 모델 결과가 다른경우, 이유를 설명해줘'  # 1턴 답을 참조하는 후속 질문
print('1:', session2.ask(q1))
print('2:', session2.ask(q2))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%
Invoking: `predict_failure_xgboost` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[XGBoost] 예측: Yes
- 고장(Failure) 확률: 96.73%
- 정상(No) 확률: 3.27%
- 주요 변수(학습 기준): Temperature(0.212), Humidity(0.140), Hours Since Previous Failure(0.110)입력값(온도 70, 습도 75, Operator2, 이전 고장 120시간)에 대해 **두 모델 결과는 서로 다릅니다.**

## 1) 로지스틱 회귀
- **예측:** No (고장 아님)
- **Failure 확률:** **11.09%**
- **No(정상) 확률:** **88.91%**

## 2) XGBoost
- **예측:** Yes (고장)
- **Failure 확률:** **96.73%**
- **No(정상) 확률:** **3.27%**
- **주요 변수(학습 기준):** Temperature(0.212), Humidity(0.140), Hours Since Previous Failure(0.110)

원하시면, “두 모델 중 어느 쪽을 더 신뢰해야 하는지”를 판단할 기준(예: 과거 성능/검증 데이터, 임계값)을 함께 정리해드릴게요.

> Finished chain.
1: 입력값(온도 70, 습도 75, 

---
## Step 6. (선택) 14.1 Agent와 합치기

RAG · 웹 검색 · ML을 한 Agent에 넣으려면 `agent_tools` 리스트에 합치면 됩니다.

### ✏️ 실습 6

1. **14.1 노트북에 정의**된 `get_current_time`, `search_pdf_tool`, `web_search`코드를 복사해 선언합니다.
2. 아래 셀에서 `agent_tools`를 채우고, `system` 프롬프트에 RAG·웹·ML Tool 용도를 각각 한 줄씩 추가하세요.
3. (선택) 합친 Agent로 *"석사 수업연한이 몇 학기야?"* 와 *"온도 70 습도 80 고장 예측"* 을 연속 질문해 보세요.

In [24]:
# ── 14.1 Tool 선언 (시각 · 주가 · PDF RAG · 웹 검색) ──
import os
from datetime import datetime

import pytz
import yfinance as yf
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import create_retriever_tool, tool
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1) 시각 · 주가
@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'


@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'


# 2) PDF RAG → search_pdf_tool
RAG_SAMPLES = os.path.join(str(WORKDIR), 'samples', 'pdf_samples')
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

all_splits = []
for pdf_name in ('학칙.pdf', '보도자료.pdf'):
    pdf_path = os.path.join(RAG_SAMPLES, pdf_name)
    if os.path.exists(pdf_path):
        all_splits.extend(text_splitter.split_documents(PyPDFLoader(pdf_path).load()))

embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
persist_directory = str(WORKDIR / 'chroma_store')

if not os.path.exists(persist_directory):
    print('Creating new Chroma store')
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory,
    )
else:
    print('Loading existing Chroma store')
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding,
    )

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

# 3) 웹 검색
web_search = DuckDuckGoSearchResults(num_results=3)

print('get_current_time:', get_current_time.name)
print('search_pdf_tool :', search_pdf_tool.name)
print('web_search      :', web_search.name)

Loading existing Chroma store
get_current_time: get_current_time
search_pdf_tool : search_pdf_documents
web_search      : duckduckgo_results_json


In [25]:
#── 14.1 + ML Tool 통합 Agent ──
agent_tools = [
    get_current_time,
    get_yf_stock_history,
    search_pdf_tool,
    web_search,
    *ML_TOOLS,
]
print('등록된 Tool:', [t.name for t in agent_tools])

combined_prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/pdf_samples PDF (학칙, 보도자료 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 현재 시각 → get_current_time
- 주가 → get_yf_stock_history
- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass

로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요.
온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

combined_agent = create_tool_calling_agent(llm, agent_tools, combined_prompt)
combined_executor = AgentExecutor(
    agent=combined_agent,
    tools=agent_tools,
    verbose=True,
    max_iterations=8,
)

등록된 Tool: ['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json', 'predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## Step 7. `input()` 대화 루프

In [ ]:
# print('종료: exit')
# while True:
#     user_input = input('사용자: ').strip()
#     if user_input.lower() in ('exit', 'quit', '종료'):
#         break
#     if not user_input:
#         continue
#     print('AI:', session.ask(user_input))

---
## 정리

| 파일 | 역할 |
|------|------|
| `ml_tools/logistic_regression_tool.py` | Day 9 로지스틱 회귀 |
| `ml_tools/xgboost_classifier_tool.py` | Day 10 XGBoost |
| `ml_tools/oneclass_svm_tool.py` | Day 11 One-Class SVM |
| `ml_tools/models/*.joblib` | 사전 학습 모델 (자동 생성) |

> ML Tool은 **추론만** 수행합니다. 재학습이 필요하면 각 `.py`를 `python logistic_regression_tool.py`처럼 직접 실행하세요.